<a href="https://colab.research.google.com/github/MZiaAfzal71/Average_Weighted_Path_Vector/blob/main/Data%20Files/Models/ANNSingleDescriptor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/MZiaAfzal71/Average_Weighted_Path_Vector.git
%cd Average_Weighted_Path_Vector/Data\ Files

Cloning into 'Average_Weighted_Path_Vector'...
remote: Enumerating objects: 778, done.
remote: Counting objects: 100% (193/193), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 778 (delta 117), reused 8 (delta 8), pack-reused 585 (from 1)
Receiving objects: 100% (778/778), 30.82 MiB | 20.99 MiB/s, done.
Resolving deltas: 100% (271/271), done.
/content/Average_Weighted_Path_Vector/Data Files


In [2]:
!pip install osfclient rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 28.7 MB/s eta 0:00:00


In [3]:
from osfclient.api import OSF
import os
from subprocess import run

# Replace with your OSF project ID
project_id = "p5ga2"   # e.g. from https://osf.io/abcd3/
osf = OSF()
project = osf.project(project_id)
store = project.storage("osfstorage")

desc_folder = []
for fold in store.folders:
    if fold.path.strip("/") == "Descriptors Data":
        desc_folder = fold
        break

# Download all files and keep folder structure
for f in desc_folder.files:
    local_path = f.path.strip("/")            # keep folders
    local_dir = os.path.dirname(local_path)   # extract dir
    if local_dir and not os.path.exists(local_dir):
        os.makedirs(local_dir, exist_ok=True) # create dirs if missing
    with open(local_path, "wb") as out:
        f.write_to(out)
    if local_path.endswith(".zip"):
      command = f"unzip '{local_path}' -d '{local_dir}'"
      run(command, shell=True)
      print(f"\nUnzipped {local_path} -> {local_dir}")
      continue
    print(f"Downloaded {f.path} -> {local_path}")

100%|██████████| 23.8M/23.8M [00:00<00:00, 24.7Mbytes/s]



Unzipped Descriptors Data/Descriptors Data.zip -> Descriptors Data


100%|██████████| 13.5M/13.5M [00:00<00:00, 18.4Mbytes/s]



Unzipped Descriptors Data/RAW_Descriptors_PWAV.zip -> Descriptors Data


In [4]:
# import pandas as pd
# import numpy as np
# import os, gc, math, random
# from dataclasses import dataclass
# from typing import List, Optional, Dict, Any, Tuple

# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# from sklearn.preprocessing import StandardScaler

# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from tqdm.auto import tqdm

import pandas as pd
import numpy as np
import os, gc, math, random, json
from dataclasses import dataclass
from typing import List, Optional, Dict, Any, Tuple

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GroupShuffleSplit

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [5]:
@dataclass
class Config:
    output_dir: str = "./desc_only_output"
    batch_size: int = 16
    epochs: int = 30
    lr: float = 1e-4
    weight_decay: float = 0.01
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    proj_dim: int = 128
    hidden_dim: int = 256
    dropout: float = 0.1

    split_mode: str = "benchmark"   # "benchmark" or "scaffold"
    test_size: float = 0.2
    n_pca: int = 64
    save_all_predictions: bool = True

def safe_name(x: str) -> str:
    return str(x).replace(" ", "").replace("/", "_").replace("\\", "_")

def murcko_scaffold(smiles):
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        return MurckoScaffold.MurckoScaffoldSmiles(mol=mol)
    except Exception:
        return None

def get_split_indices(df: pd.DataFrame, split_mode="benchmark", test_size=0.2, random_state=42):
    if split_mode == "benchmark":
        train_idx = df.index[df["Training/Test"].astype(str).str.strip().str.lower() == "training"].tolist()
        test_idx = df.index[df["Training/Test"].astype(str).str.strip().str.lower() == "test"].tolist()

        split_info = df[["SMILES"]].copy()
        if "Training/Test" in df.columns:
            split_info["split"] = df["Training/Test"]
        else:
            split_info["split"] = "unknown"
        split_info["split_mode"] = "benchmark"
        return train_idx, test_idx, split_info

    elif split_mode == "scaffold":
        if "SMILES" not in df.columns:
            raise ValueError("SMILES column is required for scaffold split.")

        scaffold_df = df[["SMILES"]].copy()
        scaffold_df["scaffold"] = scaffold_df["SMILES"].apply(murcko_scaffold)

        valid_mask = scaffold_df["scaffold"].notna()
        valid_idx = scaffold_df.index[valid_mask]
        groups = scaffold_df.loc[valid_idx, "scaffold"]

        if len(valid_idx) == 0:
            raise ValueError("No valid Murcko scaffolds could be generated.")

        gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
        train_pos, test_pos = next(gss.split(valid_idx, groups=groups))

        train_idx = valid_idx[train_pos].tolist()
        test_idx = valid_idx[test_pos].tolist()

        split_info = scaffold_df.copy()
        split_info["split"] = "unused"
        split_info.loc[train_idx, "split"] = "Training"
        split_info.loc[test_idx, "split"] = "Test"
        split_info["split_mode"] = "scaffold"

        return train_idx, test_idx, split_info

    else:
        raise ValueError(f"Unknown split_mode: {split_mode}")

def get_feature_columns(df: pd.DataFrame, prop: str, desc_name: str):
    target_col = f"{prop}-Measured"
    all_cols = list(df.columns)

    meta_cols = {
        "Preferred_name", "SMILES", "Training/Test", f"{prop}-EPISuite Prediction",
        f"{prop}-Prediction from our model", "CAS RN",	"NAME",	"IUPAC Name",
        target_col
    }

    if desc_name in {"PWAV_raw", "PWAV_pruned"}:
        pwav_cols = [c for c in all_cols if c.startswith(f"{prop}_PWAV_")]
        extra_cols = [c for c in all_cols if c.startswith(f"{prop}_EXTRA_")]
        return {
            "target_col": target_col,
            "pwav_cols": pwav_cols,
            "extra_cols": extra_cols,
            "direct_feature_cols": None
        }
    else:
        direct_feature_cols = [c for c in all_cols if c not in meta_cols]
        return {
            "target_col": target_col,
            "pwav_cols": None,
            "extra_cols": None,
            "direct_feature_cols": direct_feature_cols
        }

def prepare_descriptor_arrays(
    df: pd.DataFrame,
    prop: str,
    desc_name: str,
    train_idx: List[int],
    test_idx: List[int],
    cfg: Config
):
    col_info = get_feature_columns(df, prop, desc_name)
    target_col = col_info["target_col"]

    train_df = df.loc[train_idx].reset_index(drop=True).copy()
    test_df = df.loc[test_idx].reset_index(drop=True).copy()

    y_train = train_df[target_col].to_numpy(dtype=np.float32)
    y_test = test_df[target_col].to_numpy(dtype=np.float32)

    pca_model = None

    if desc_name in {"PWAV_raw", "PWAV_pruned"}:
        pwav_cols = col_info["pwav_cols"]
        extra_cols = col_info["extra_cols"]

        X_train_core = train_df[pwav_cols].to_numpy(dtype=np.float32)
        X_test_core = test_df[pwav_cols].to_numpy(dtype=np.float32)

        if cfg.n_pca is not None and cfg.n_pca < X_train_core.shape[1]:
            pca_model = PCA(n_components=cfg.n_pca, random_state=cfg.seed)
            X_train_core = pca_model.fit_transform(X_train_core)
            X_test_core = pca_model.transform(X_test_core)

        if extra_cols:
            X_train_extra = train_df[extra_cols].to_numpy(dtype=np.float32)
            X_test_extra = test_df[extra_cols].to_numpy(dtype=np.float32)
            X_train = np.hstack([X_train_core, X_train_extra]).astype(np.float32)
            X_test = np.hstack([X_test_core, X_test_extra]).astype(np.float32)
        else:
            X_train = X_train_core.astype(np.float32)
            X_test = X_test_core.astype(np.float32)

    else:
        direct_cols = col_info["direct_feature_cols"]
        X_train = train_df[direct_cols].to_numpy(dtype=np.float32)
        X_test = test_df[direct_cols].to_numpy(dtype=np.float32)

    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train).astype(np.float32)
    X_test_scaled = scaler.transform(X_test).astype(np.float32)

    return {
        "train_df": train_df,
        "test_df": test_df,
        "X_train": X_train_scaled,
        "X_test": X_test_scaled,
        "y_train": y_train,
        "y_test": y_test,
        "scaler": scaler,
        "pca_model": pca_model,
        "col_info": col_info
    }

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

# ----------------------------
# Model (Single descriptor)
# ----------------------------
class DescriptorSingle(nn.Module):
    def __init__(self, n_desc: int, proj_dim: int = 128, hidden: int = 256, dropout: float = 0.1):
        super().__init__()
        self.desc_proj = nn.Sequential(
            nn.Linear(n_desc, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(proj_dim, hidden),
            nn.LayerNorm(hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.regressor = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, desc):
        h = self.desc_proj(desc)
        return self.regressor(h).squeeze(-1)

# ----------------------------
# Dataset / Collate
# ----------------------------
class DescDataset(Dataset):
    def __init__(self, targets: Optional[np.ndarray], descriptors: Optional[np.ndarray] = None):
        self.targets = None if targets is None else np.asarray(targets, dtype=np.float32)
        self.desc = None if descriptors is None else np.asarray(descriptors, dtype=np.float32)

    def __len__(self):
        return len(self.targets) if self.targets is not None else len(self.desc)

    def __getitem__(self, i):
        item = {}
        if self.targets is not None:
            item["labels"] = torch.tensor(self.targets[i], dtype=torch.float32)
        if self.desc is not None:
            d = self.desc[i]
            if not np.all(np.isfinite(d)):
                d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)
            item["descriptors"] = torch.tensor(d, dtype=torch.float32)
        return item

def collate_stack(batch):
    out = {}
    if "labels" in batch[0]:
        out["labels"] = torch.stack([b["labels"] for b in batch])
    if "descriptors" in batch[0]:
        out["descriptors"] = torch.stack([b["descriptors"] for b in batch])
    return out

def make_loaders_from_arrays(X_train, y_train, X_test, y_test, cfg: Config):
    train_ds = DescDataset(y_train, X_train)
    test_ds = DescDataset(y_test, X_test)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_stack
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_stack
    )
    return train_loader, test_loader

def train_single_desc_for_prop(prop: str, desc_name: str, cfg: Config) -> Dict[str, Any]:
    set_seed(cfg.seed)
    ensure_dir(cfg.output_dir)

    safe_prop = safe_name(prop)
    data_file = f"Descriptors Data/{safe_prop}_{desc_name}.parquet"
    run_name = f"{safe_prop}_{desc_name}_{cfg.split_mode}"

    df = pd.read_parquet(data_file)

    split_train_idx, split_test_idx, split_info = get_split_indices(
        df,
        split_mode=cfg.split_mode,
        test_size=cfg.test_size,
        random_state=cfg.seed
    )

    prep = prepare_descriptor_arrays(
        df=df,
        prop=prop,
        desc_name=desc_name,
        train_idx=split_train_idx,
        test_idx=split_test_idx,
        cfg=cfg
    )

    train_loader, test_loader = make_loaders_from_arrays(
        prep["X_train"], prep["y_train"],
        prep["X_test"], prep["y_test"],
        cfg
    )

    n_desc = prep["X_train"].shape[1]
    model = DescriptorSingle(
        n_desc=n_desc,
        proj_dim=cfg.proj_dim,
        hidden=cfg.hidden_dim,
        dropout=cfg.dropout
    ).to(cfg.device)

    optim = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    best_mse = float("inf")
    best_path = os.path.join(cfg.output_dir, f"{run_name}_best.pt")

    for epoch in tqdm(range(1, cfg.epochs + 1), desc=f"{run_name} epochs"):
        model.train()
        ep_loss = 0.0

        for batch in train_loader:
            batch = {k: v.to(cfg.device) for k, v in batch.items()}
            pred = model(batch["descriptors"])
            loss = F.mse_loss(pred, batch["labels"])

            optim.zero_grad()
            loss.backward()
            optim.step()

            ep_loss += loss.item()

        print(f"Epoch {epoch}/{cfg.epochs} | train MSE: {ep_loss / max(1, len(train_loader)):.6f}")

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(cfg.device) for k, v in batch.items()}
                out = model(batch["descriptors"])
                preds.extend(out.detach().cpu().numpy())
                labels.extend(batch["labels"].detach().cpu().numpy())

        mse = mean_squared_error(labels, preds)
        print(f"→ Test MSE: {mse:.6f} | RMSE: {math.sqrt(mse):.6f}")

        if mse < best_mse:
            best_mse = mse
            torch.save(model.state_dict(), best_path)
            print(f"  ✓ Saved best checkpoint → {best_path}")

    # Load best checkpoint before final evaluation
    model.load_state_dict(torch.load(best_path, map_location=cfg.device))
    model.eval()

    # Final test predictions
    final_test_preds = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(cfg.device) for k, v in batch.items()}
            out = model(batch["descriptors"])
            final_test_preds.extend(out.detach().cpu().numpy())

    y_test = prep["y_test"]
    mae_v = mean_absolute_error(y_test, final_test_preds)
    rmse_v = rmse(y_test, final_test_preds)
    r2_v = r2_score(y_test, final_test_preds)

    print(f"Final Test metrics for {run_name} → MAE: {mae_v:.4f} | RMSE: {rmse_v:.4f} | R²: {r2_v:.4f}")

    # Save split assignments
    split_path = os.path.join(cfg.output_dir, f"{run_name}_split.csv")
    split_info.to_csv(split_path, index=False)

    # Save PCA metadata if applicable
    if prep["pca_model"] is not None:
        pca_meta = {
            "property": prop,
            "descriptor": desc_name,
            "split_mode": cfg.split_mode,
            "n_components": int(prep["pca_model"].n_components_),
            "explained_variance_ratio_sum": float(np.sum(prep["pca_model"].explained_variance_ratio_))
        }
        pca_path = os.path.join(cfg.output_dir, f"{run_name}_pca.json")
        with open(pca_path, "w") as f:
            json.dump(pca_meta, f, indent=2)

    # Save test predictions
    test_df = prep["test_df"].copy()
    test_results = pd.DataFrame({
        "Preferred_name": test_df["Preferred_name"] if "Preferred_name" in test_df.columns else pd.Series([None] * len(test_df)),
        "SMILES": test_df["SMILES"],
        "Observed": y_test,
        "Predicted": final_test_preds,
        "split_mode": cfg.split_mode
    })
    test_pred_path = os.path.join(cfg.output_dir, f"{run_name}_test_predictions.parquet")
    test_results.to_parquet(test_pred_path, index=False)

    # Optional all-row predictions
    if cfg.save_all_predictions:
        col_info = prep["col_info"]

        if desc_name in {"PWAV_raw", "PWAV_pruned"}:
            pwav_cols = col_info["pwav_cols"]
            extra_cols = col_info["extra_cols"]

            X_all_core = df[pwav_cols].to_numpy(dtype=np.float32)
            if prep["pca_model"] is not None:
                X_all_core = prep["pca_model"].transform(X_all_core)

            if extra_cols:
                X_all_extra = df[extra_cols].to_numpy(dtype=np.float32)
                X_all = np.hstack([X_all_core, X_all_extra]).astype(np.float32)
            else:
                X_all = X_all_core.astype(np.float32)
        else:
            X_all = df[col_info["direct_feature_cols"]].to_numpy(dtype=np.float32)

        X_all = prep["scaler"].transform(X_all).astype(np.float32)
        all_ds = DescDataset(df[f"{prop}-Measured"].to_numpy(dtype=np.float32), X_all)
        all_loader = DataLoader(all_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_stack)

        all_preds = []
        with torch.no_grad():
            for batch in all_loader:
                batch = {k: v.to(cfg.device) for k, v in batch.items()}
                out = model(batch["descriptors"])
                all_preds.extend(out.detach().cpu().numpy())

        all_results = pd.DataFrame({
            "Preferred_name": df["Preferred_name"] if "Preferred_name" in df.columns else pd.Series([None] * len(df)),
            "SMILES": df["SMILES"],
            "Observed": df[f"{prop}-Measured"],
            "Predicted": all_preds,
            "Training/Test": df["Training/Test"] if "Training/Test" in df.columns else pd.Series([None] * len(df)),
            "split_mode": cfg.split_mode
        })

        all_pred_path = os.path.join(cfg.output_dir, f"{run_name}_all_predictions.parquet")
        all_results.to_parquet(all_pred_path, index=False)

    return {
        "MAE": mae_v,
        "RMSE": rmse_v,
        "R2": r2_v,
        "n_train": len(split_train_idx),
        "n_test": len(split_test_idx),
        "descriptor_dim": int(n_desc)
    }

In [ ]:
# # ----------------------------
# # Config
# # ----------------------------
# @dataclass
# class Config:
#     output_dir: str = "./desc_only_output"
#     batch_size: int = 16
#     epochs: int = 5
#     lr: float = 1e-5
#     weight_decay: float = 0.01
#     seed: int = 42
#     device: str = "cuda" if torch.cuda.is_available() else "cpu"
#     proj_dim: int = 128
#     dropout: float = 0.1

# # ----------------------------
# # Utils
# # ----------------------------
# def set_seed(seed: int):
#     random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
#     if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# def rmse(y_true, y_pred):
#     return float(np.sqrt(mean_squared_error(y_true, y_pred)))

# def ensure_dir(path: str):
#     os.makedirs(path, exist_ok=True)

# # ----------------------------
# # Model (Single descriptor)
# # ----------------------------
# class DescriptorSingle(nn.Module):
#     def __init__(self, n_desc: int, proj_dim: int = 128, hidden: int = 256, dropout: float = 0.1):
#         super().__init__()
#         self.desc_proj = nn.Sequential(
#             nn.Linear(n_desc, proj_dim),
#             nn.LayerNorm(proj_dim),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(proj_dim, hidden),
#             nn.LayerNorm(hidden),
#             nn.ReLU(),
#         )
#         self.dropout = nn.Dropout(dropout)
#         self.regressor = nn.Sequential(
#             nn.Linear(hidden, hidden // 2),
#             nn.ReLU(),
#             nn.Dropout(dropout),
#             nn.Linear(hidden // 2, 1),
#         )

#     def forward(self, desc):
#         h = self.desc_proj(desc)
#         h = self.dropout(h)
#         return self.regressor(h).squeeze(-1)

# # ----------------------------
# # Dataset / Collate
# # ----------------------------
# class DescDataset(Dataset):
#     def __init__(self, targets: Optional[np.ndarray], descriptors: Optional[np.ndarray] = None):
#         self.targets = None if targets is None else np.asarray(targets, dtype=np.float32)
#         self.desc = None if descriptors is None else np.asarray(descriptors, dtype=np.float32)

#     def __len__(self):
#         return len(self.targets) if self.targets is not None else len(self.desc)

#     def __getitem__(self, i):
#         item = {}
#         if self.targets is not None:
#             item["labels"] = torch.tensor(self.targets[i], dtype=torch.float32)
#         if self.desc is not None:
#             d = self.desc[i]
#             if not np.all(np.isfinite(d)):
#                 d = np.nan_to_num(d, nan=0.0, posinf=0.0, neginf=0.0)
#             item["descriptors"] = torch.tensor(d, dtype=torch.float32)
#         return item

# def collate_stack(batch):
#     out = {}
#     if "labels" in batch[0]:
#         out["labels"] = torch.stack([b["labels"] for b in batch])
#     if "descriptors" in batch[0]:
#         out["descriptors"] = torch.stack([b["descriptors"] for b in batch])
#     return out

# # ----------------------------
# # Training / Evaluation
# # ----------------------------
# def make_loaders(df: pd.DataFrame, target_col: str, cfg: Config,
#                  desc_cols: List[str]) -> Tuple[DataLoader, DataLoader, StandardScaler]:
#     train_df = df[df["Training/Test"].str.strip().str.lower() == "training"].reset_index(drop=True)
#     test_df  = df[df["Training/Test"].str.strip().str.lower() == "test"].reset_index(drop=True)

#     scaler = StandardScaler().fit(train_df[desc_cols].to_numpy(dtype=np.float32))
#     train_desc = scaler.transform(train_df[desc_cols].to_numpy(dtype=np.float32))
#     test_desc  = scaler.transform(test_df[desc_cols].to_numpy(dtype=np.float32))

#     train_ds = DescDataset(train_df[target_col].to_numpy(dtype=np.float32), train_desc)
#     test_ds  = DescDataset(test_df[target_col].to_numpy(dtype=np.float32), test_desc)

#     train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_stack)
#     test_loader  = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_stack)
#     return train_loader, test_loader, scaler


# def train_single_desc_for_prop(prop: str, desc_name: str, cfg: Config) -> Dict[str, Any]:
#     set_seed(cfg.seed)
#     ensure_dir(cfg.output_dir)

#     target_col = f"{prop}-Measured"
#     data_file = f"Descriptors Data/{prop}_{desc_name}.parquet"
#     sheet_name = f"{prop}_{desc_name}"

#     df = pd.read_parquet(data_file)
#     desc_cols = df.columns[9:].to_list()

#     # Data loaders
#     train_loader, test_loader, scaler = make_loaders(df, target_col, cfg, desc_cols)

#     # Model
#     n_desc = len(desc_cols)
#     model = DescriptorSingle(n_desc, proj_dim=cfg.proj_dim, dropout=cfg.dropout).to(cfg.device)
#     optim = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

#     best_mse = float("inf")
#     best_path = os.path.join(cfg.output_dir, f"{sheet_name}_best.pt")

#     for epoch in tqdm(range(1, cfg.epochs + 1)):
#         model.train(); ep_loss = 0.0
#         for batch in tqdm(train_loader):
#             batch = {k: v.to(cfg.device) for k, v in batch.items()}
#             pred = model(batch["descriptors"])
#             loss = F.mse_loss(pred, batch["labels"])
#             optim.zero_grad(); loss.backward(); optim.step()
#             ep_loss += loss.item()
#         print(f"Epoch {epoch}/{cfg.epochs} | train MSE: {ep_loss / max(1,len(train_loader)):.6f}")

#         model.eval(); preds, labels = [], []
#         with torch.no_grad():
#             for batch in test_loader:
#                 batch = {k: v.to(cfg.device) for k, v in batch.items()}
#                 out = model(batch["descriptors"])
#                 preds.extend(out.detach().cpu().numpy())
#                 labels.extend(batch["labels"].detach().cpu().numpy())
#         mse = mean_squared_error(labels, preds)
#         print(f"→ Test MSE: {mse:.6f} | RMSE: {math.sqrt(mse):.6f}")
#         if mse < best_mse:
#             best_mse = mse
#             torch.save(model.state_dict(), best_path)
#             print(f"  ✓ Saved best checkpoint → {best_path}")

#     model.eval()

#     all_desc  = scaler.transform(df[desc_cols].to_numpy(dtype=np.float32))
#     all_ds = DescDataset(df[target_col].to_numpy(dtype=np.float32), all_desc)

#     all_loader = DataLoader(all_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_stack)

#     all_preds = []
#     with torch.no_grad():
#         for batch in all_loader:
#             batch = {k: v.to(cfg.device) for k, v in batch.items()}
#             out = model(batch["descriptors"])
#             all_preds.extend(out.detach().cpu().numpy())

#     # Build results DF
#     new_results = pd.DataFrame({
#         "Name": df["NAME"] if "NAME" in df.columns else pd.Series([None]*len(df)),
#         "SMILES": df["SMILES"],
#         "Observed": df[target_col],
#         "Predicted": all_preds,
#         "Training/Test": df["Training/Test"],
#     })

#     # Final metrics on Test only
#     obs_test = new_results[new_results["Training/Test"].str.lower() == "test"]["Observed"].values
#     pred_test = new_results[new_results["Training/Test"].str.lower() == "test"]["Predicted"].values
#     mae_v = mean_absolute_error(obs_test, pred_test)
#     rmse_v = rmse(obs_test, pred_test)
#     r2_v = r2_score(obs_test, pred_test)
#     print(f"Final Test metrics for {sheet_name} → MAE: {mae_v:.4f} | RMSE: {rmse_v:.4f} | R²: {r2_v:.4f}")

#     # Save predictions parquet
#     pred_path = os.path.join(cfg.output_dir, f"{sheet_name}.parquet")
#     new_results.to_parquet(pred_path, index=False)
#     print(f"Saved predictions → {pred_path}")

#     return {
#         "MAE": mae_v, "RMSE": rmse_v, "R2": r2_v,
#     }


# # ----------------------------
# # Multi-property runner
# # ----------------------------
# def run_all_properties_single_desc(prop_names: List[str], desc_name: str, cfg: Config):
#     ensure_dir(cfg.output_dir)
#     perf_rows = []
#     for prop in prop_names:
#         print(f"\n=== Processing: {prop}_{desc_name} ===")
#         result = train_single_desc_for_prop(prop, desc_name, cfg)
#         perf_rows.append([f"{prop}_{desc_name}", result["MAE"], result["RMSE"], result["R2"]])
#     perf_df = pd.DataFrame(perf_rows, columns=["Property", "MAE", "RMSE", "R2"])
#     stats_path = os.path.join(cfg.output_dir, f"{desc_name}_single_stats.csv")
#     perf_df.to_csv(stats_path, index=False)
#     print(f"\n📊 Stats saved → {stats_path}")
#     return perf_df


In [6]:
def run_all_properties_single_desc(prop_names: List[str], desc_name: str, cfg: Config):
    ensure_dir(cfg.output_dir)
    perf_rows = []

    for prop in prop_names:
        print(f"\n=== Processing: {prop}_{desc_name}_{cfg.split_mode} ===")
        result = train_single_desc_for_prop(prop, desc_name, cfg)

        perf_rows.append({
            "property": prop,
            "descriptor": desc_name,
            "split_mode": cfg.split_mode,
            "n_train": result["n_train"],
            "n_test": result["n_test"],
            "descriptor_dim": result["descriptor_dim"],
            "MAE": result["MAE"],
            "RMSE": result["RMSE"],
            "R2": result["R2"]
        })

    perf_df = pd.DataFrame(perf_rows)
    stats_path = os.path.join(cfg.output_dir, f"{desc_name}_{cfg.split_mode}_single_stats.csv")
    perf_df.to_csv(stats_path, index=False)

    print(f"\n📊 Stats saved → {stats_path}")
    return perf_df

In [8]:
cfg = Config(
    output_dir="./desc_only_output",
    epochs=30,
    batch_size=8,
    proj_dim=128,
    hidden_dim=256,
    lr=1e-4,
    dropout=0.1,
    split_mode="scaffold",   # change to "scaffold" for scaffold experiments
    n_pca=64,
    seed=42
)

prop_names = ["Log VP", "MP", "BP", "LogBCF", "LogS", "LogP"]

# Use names matching your saved descriptor files
desc_names = ["MACCS", "Morgan", "PWAV_raw"]
# Later you can add "PWAV_pruned"

for desc in desc_names:
    perf_df = run_all_properties_single_desc(prop_names, desc, cfg)
    print(perf_df)


=== Processing: Log VP_MACCS_scaffold ===


LogVP_MACCS_scaffold epochs:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1/30 | train MSE: 13.075460
→ Test MSE: 15.046892 | RMSE: 3.879032
  ✓ Saved best checkpoint → ./desc_only_output/LogVP_MACCS_scaffold_best.pt
Epoch 2/30 | train MSE: 5.483827
→ Test MSE: 7.086051 | RMSE: 2.661964
  ✓ Saved best checkpoint → ./desc_only_output/LogVP_MACCS_scaffold_best.pt
Epoch 3/30 | train MSE: 3.428973
→ Test MSE: 5.779009 | RMSE: 2.403957
  ✓ Saved best checkpoint → ./desc_only_output/LogVP_MACCS_scaffold_best.pt
Epoch 4/30 | train MSE: 2.727418
→ Test MSE: 5.124397 | RMSE: 2.263713
  ✓ Saved best checkpoint → ./desc_only_output/LogVP_MACCS_scaffold_best.pt
Epoch 5/30 | train MSE: 2.360288
→ Test MSE: 5.014971 | RMSE: 2.239413
  ✓ Saved best checkpoint → ./desc_only_output/LogVP_MACCS_scaffold_best.pt
Epoch 6/30 | train MSE: 2.118224
→ Test MSE: 4.803297 | RMSE: 2.191643
  ✓ Saved best checkpoint → ./desc_only_output/LogVP_MACCS_scaffold_best.pt
Epoch 7/30 | train MSE: 1.905505
→ Test MSE: 4.649252 | RMSE: 2.156212
  ✓ Saved best checkpoint → ./desc_only_outpu

MP_MACCS_scaffold epochs:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1/30 | train MSE: 9465.283056
→ Test MSE: 4513.903754 | RMSE: 67.185592
  ✓ Saved best checkpoint → ./desc_only_output/MP_MACCS_scaffold_best.pt
Epoch 2/30 | train MSE: 3658.881353
→ Test MSE: 3468.348432 | RMSE: 58.892686
  ✓ Saved best checkpoint → ./desc_only_output/MP_MACCS_scaffold_best.pt
Epoch 3/30 | train MSE: 2906.334121
→ Test MSE: 2936.020522 | RMSE: 54.185058
  ✓ Saved best checkpoint → ./desc_only_output/MP_MACCS_scaffold_best.pt
Epoch 4/30 | train MSE: 2605.230784
→ Test MSE: 2797.806453 | RMSE: 52.894295
  ✓ Saved best checkpoint → ./desc_only_output/MP_MACCS_scaffold_best.pt
Epoch 5/30 | train MSE: 2418.095462
→ Test MSE: 2683.007661 | RMSE: 51.797757
  ✓ Saved best checkpoint → ./desc_only_output/MP_MACCS_scaffold_best.pt
Epoch 6/30 | train MSE: 2292.207540
→ Test MSE: 2738.471795 | RMSE: 52.330410
Epoch 7/30 | train MSE: 2219.488827
→ Test MSE: 2560.258977 | RMSE: 50.599002
  ✓ Saved best checkpoint → ./desc_only_output/MP_MACCS_scaffold_best.pt
Epoch 8/30 | tra

...

Epoch 1/30 | train MSE: 4.701326
→ Test MSE: 1.087912 | RMSE: 1.043030
  ✓ Saved best checkpoint → ./desc_only_output/LogBCF_PWAV_raw_scaffold_best.pt
Epoch 2/30 | train MSE: 1.993927
→ Test MSE: 1.075410 | RMSE: 1.037020
  ✓ Saved best checkpoint → ./desc_only_output/LogBCF_PWAV_raw_scaffold_best.pt
Epoch 3/30 | train MSE: 1.333470
→ Test MSE: 0.898405 | RMSE: 0.947842
  ✓ Saved best checkpoint → ./desc_only_output/LogBCF_PWAV_raw_scaffold_best.pt
Epoch 4/30 | train MSE: 0.976045
→ Test MSE: 0.653961 | RMSE: 0.808679
  ✓ Saved best checkpoint → ./desc_only_output/LogBCF_PWAV_raw_scaffold_best.pt
Epoch 5/30 | train MSE: 0.694618
→ Test MSE: 0.579090 | RMSE: 0.760980
  ✓ Saved best checkpoint → ./desc_only_output/LogBCF_PWAV_raw_scaffold_best.pt
Epoch 6/30 | train MSE: 0.552633
→ Test MSE: 0.542094 | RMSE: 0.736270
  ✓ Saved best checkpoint → ./desc_only_output/LogBCF_PWAV_raw_scaffold_best.pt
Epoch 7/30 | train MSE: 0.462165
→ Test MSE: 0.532717 | RMSE: 0.729875
  ✓ Saved best checkpoi

LogS_PWAV_raw_scaffold epochs:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1/30 | train MSE: 3.249785
→ Test MSE: 3.192273 | RMSE: 1.786693
  ✓ Saved best checkpoint → ./desc_only_output/LogS_PWAV_raw_scaffold_best.pt
Epoch 2/30 | train MSE: 1.128005
→ Test MSE: 1.844454 | RMSE: 1.358107
  ✓ Saved best checkpoint → ./desc_only_output/LogS_PWAV_raw_scaffold_best.pt
Epoch 3/30 | train MSE: 0.866631
→ Test MSE: 1.523562 | RMSE: 1.234327
  ✓ Saved best checkpoint → ./desc_only_output/LogS_PWAV_raw_scaffold_best.pt
Epoch 4/30 | train MSE: 0.730143
→ Test MSE: 1.201218 | RMSE: 1.096001
  ✓ Saved best checkpoint → ./desc_only_output/LogS_PWAV_raw_scaffold_best.pt
Epoch 5/30 | train MSE: 0.606084
→ Test MSE: 1.098441 | RMSE: 1.048066
  ✓ Saved best checkpoint → ./desc_only_output/LogS_PWAV_raw_scaffold_best.pt
Epoch 6/30 | train MSE: 0.559106
→ Test MSE: 0.980713 | RMSE: 0.990309
  ✓ Saved best checkpoint → ./desc_only_output/LogS_PWAV_raw_scaffold_best.pt
Epoch 7/30 | train MSE: 0.541000
→ Test MSE: 0.922186 | RMSE: 0.960305
  ✓ Saved best checkpoint → ./desc_

LogP_PWAV_raw_scaffold epochs:   0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1/30 | train MSE: 1.420667
→ Test MSE: 0.910112 | RMSE: 0.953998
  ✓ Saved best checkpoint → ./desc_only_output/LogP_PWAV_raw_scaffold_best.pt
Epoch 2/30 | train MSE: 0.713323
→ Test MSE: 0.816739 | RMSE: 0.903736
  ✓ Saved best checkpoint → ./desc_only_output/LogP_PWAV_raw_scaffold_best.pt
Epoch 3/30 | train MSE: 0.576387
→ Test MSE: 0.708274 | RMSE: 0.841590
  ✓ Saved best checkpoint → ./desc_only_output/LogP_PWAV_raw_scaffold_best.pt
Epoch 4/30 | train MSE: 0.504441
→ Test MSE: 0.655843 | RMSE: 0.809841
  ✓ Saved best checkpoint → ./desc_only_output/LogP_PWAV_raw_scaffold_best.pt
Epoch 5/30 | train MSE: 0.446815
→ Test MSE: 0.615854 | RMSE: 0.784764
  ✓ Saved best checkpoint → ./desc_only_output/LogP_PWAV_raw_scaffold_best.pt
Epoch 6/30 | train MSE: 0.412543
→ Test MSE: 0.620394 | RMSE: 0.787651
Epoch 7/30 | train MSE: 0.374975
→ Test MSE: 0.630894 | RMSE: 0.794288
Epoch 8/30 | train MSE: 0.360721
→ Test MSE: 0.622211 | RMSE: 0.788803
Epoch 9/30 | train MSE: 0.338243
→ Test MS

In [ ]:
# cfg = Config(
#     epochs=30,
#     batch_size=8,
#     proj_dim=128,
#     lr = 1e-4
# )

# prop_names = ["Log VP", "MP", "BP", "LogBCF", "LogS", "LogP"]
# # prop_names = ["LogBCF"]

# desc_names = ["MACCS", "Morgan", "pwav"]
# # desc_names = ["pwav"]

# for desc in desc_names:
#     perf_df = run_all_properties_single_desc(prop_names, desc, cfg)
#     print(perf_df)


In [9]:
!zip -r desc_only_output.zip desc_only_output/

  adding: desc_only_output/ (stored 0%)
  adding: desc_only_output/LogS_PWAV_raw_scaffold_test_predictions.parquet (deflated 28%)
  adding: desc_only_output/LogS_MACCS_benchmark_all_predictions.parquet (deflated 12%)
  adding: desc_only_output/MP_MACCS_scaffold_all_predictions.parquet (deflated 9%)
  adding: desc_only_output/MP_MACCS_benchmark_all_predictions.parquet (deflated 9%)
  adding: desc_only_output/MP_PWAV_raw_scaffold_best.pt (deflated 8%)
  adding: desc_only_output/MP_Morgan_benchmark_best.pt (deflated 8%)
  adding: desc_only_output/MP_Morgan_scaffold_test_predictions.parquet (deflated 15%)
  adding: desc_only_output/LogS_MACCS_scaffold_test_predictions.parquet (deflated 28%)
  adding: desc_only_output/LogVP_MACCS_benchmark_best.pt (deflated 9%)
  adding: desc_only_output/BP_Morgan_scaffold_test_predictions.parquet (deflated 10%)
  adding: desc_only_output/LogVP_PWAV_raw_scaffold_test_predictions.parquet (deflated 15%)
  adding: desc_only_output/LogP_Morgan_benchmark_all_pre